# 01 — Download raw data (PRIDE PXD029845)

Kim et al. 2022, *Transl Psychiatry* 12:142 — prefrontal cortex (BA9/10), TMT 10-plex, 23 suicide decedents and 13 sudden-death controls.

The file list, sizes and SHA-1 checksums come from the PRIDE API and are stored in `config/PXD029845_manifest.tsv`.
Downloads resume after interruption and every file is verified against its checksum. Files go to `data_raw/`, which is excluded from version control.

In [ ]:
import hashlib, time
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data_raw"          # change here to place raw files on another disk
RAW.mkdir(exist_ok=True)

manifest = pd.read_csv(ROOT / "config" / "PXD029845_manifest.tsv", sep="\t")
manifest.groupby("category")["size_bytes"].agg(files="count", gb=lambda s: round(s.sum() / 1e9, 1))

## What to download

`RAW` (96 files) and the FASTA file are needed. The Proteome Discoverer result files (`.msf`, 87 GB; `.pep.xml`, 2 GB) are not needed for a FragPipe re-analysis.

In [ ]:
wanted = manifest[(manifest["category"] == "RAW") | manifest["file_name"].str.lower().str.endswith((".fasta", ".txt"))].reset_index(drop=True)
print(len(wanted), "files,", round(wanted["size_bytes"].sum() / 1e9, 1), "GB")

In [ ]:
def sha1_of(path, block=1 << 22):
    h = hashlib.sha1()
    with open(path, "rb") as fh:
        while chunk := fh.read(block):
            h.update(chunk)
    return h.hexdigest()

def download(row, retries=5):
    target = RAW / row.file_name
    for attempt in range(retries):
        have = target.stat().st_size if target.exists() else 0
        if have == row.size_bytes:
            break
        if have > row.size_bytes:
            target.unlink(); have = 0
        try:
            with requests.get(row.url, headers={"Range": f"bytes={have}-"}, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(target, "ab") as fh, tqdm(total=row.size_bytes, initial=have, unit="B", unit_scale=True, desc=row.file_name, leave=False) as bar:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk); bar.update(len(chunk))
        except requests.RequestException as exc:
            print(row.file_name, "retry", attempt + 1, type(exc).__name__)
            time.sleep(10)
    return target.exists() and target.stat().st_size == row.size_bytes

In [ ]:
log_path = ROOT / "results" / "download_log.tsv"
done = set(pd.read_csv(log_path, sep="\t").query("status == 'verified'")["file_name"]) if log_path.exists() else set()
records = []
for row in tqdm(list(wanted.itertuples()), desc="files"):
    if row.file_name in done:
        records.append((row.file_name, "verified")); continue
    ok = download(row)
    status = "verified" if ok and sha1_of(RAW / row.file_name) == row.sha1 else "FAILED"
    records.append((row.file_name, status))
    pd.DataFrame(records, columns=["file_name", "status"]).to_csv(log_path, sep="\t", index=False)
pd.DataFrame(records, columns=["file_name", "status"])["status"].value_counts()